# Multimodal biometrics v2 — optimized Kaggle T4 run

Attach **SOCOFing** and **LFW deep-funneled**. Select **GPU T4**, enable **Internet** for pretrained weights, then **Restart & Run All**.

Setup checks PyTorch's core APIs and CUDA synchronization before image hashing. If `torch.device` is missing or the import is stale, restart the Kaggle session and run from the installation cell with Kaggle's installed torch/torchvision stack. Inventory hashing and calibration record CPU wall time without a CUDA wait; GPU stages retain synchronization. See [PyTorch's device API](https://docs.pytorch.org/docs/2.11/tensor_attributes.html#torch.device) and [CUDA synchronization](https://docs.pytorch.org/docs/2.11/generated/torch.cuda.synchronize.html).

This preserves v2's identity splits, models, losses, checkpoint rule, fusion selection, and two threshold policies. It adds ordered threaded image hashing, bounded decoded-image caching, worker-free loading, encoder reuse, 2,000 bootstrap replicates, and matched conservative comparisons. Worker changes alter the augmentation random stream; results must be measured in this new run.

The first run freshly hashes pixels (`STRICT_INVENTORY_REHASH=True`). For a later run on the **same immutable Kaggle dataset version**, attach the generated inventory-cache ZIP as an extracted Kaggle dataset, set `INVENTORY_CACHE_READ_DIR` to its directory and set `STRICT_INVENTORY_REHASH=False`. Cache reuse validates filenames, sizes and modification times, and records that pixels were not freshly rehashed. Keep strict mode for changed or uncertain data.

`NUM_WORKERS=0` avoids the cleanup errors in the previous notebook; the main-process image cache is capped at 512 MiB. `INVENTORY_IO_WORKERS=4` controls hashing threads independently. Use 1 if the Kaggle input mount slows under concurrent reads. Timing improvements are unmeasured until this notebook runs on Kaggle.

Download the executed notebook, the new run ZIP, and the separate inventory-cache ZIP. Treat this reused test cohort as exploratory; do not choose thresholds or training epochs using its test results.

**Evidence and research:** [PyTorch 2.11 data loading](https://docs.pytorch.org/docs/2.11/data.html) explains the small in-memory dataset case and the tradeoff with asynchronous workers. The [performance guide](https://docs.pytorch.org/tutorials/recipes/recipes/tuning_guide.html) recommends measuring worker settings for the actual workload. [Python's thread-pool documentation](https://docs.python.org/3.13/library/concurrent.futures.html) supports overlapping I/O; the chosen four-thread setting is a hypothesis to measure. [PyTorch AMP guidance](https://docs.pytorch.org/docs/2.11/notes/amp_examples.html) supports the preserved unscale/clip/skip sequence. [NIST's operational ROC study](https://www.nist.gov/publications/measures-uncertainties-and-significance-test-operational-roc-analysis) distinguishes EER from rates at a fixed threshold and considers correlated comparisons; it does not validate this notebook's exact bootstrap implementation. Sources checked 11 November 2026.

Inspect the exported `stage_timings.csv` alongside epoch histories. Compare complete runs with the same immutable dataset, input-cache state, Kaggle image and T4 allocation; kernel timing alone is not an end-to-end speed measurement. The 2,000-replicate bootstrap deliberately adds evaluation work to improve interval stability.


# Multimodal biometric verification — selection and calibration v2

**Kaggle T4 · LFW + SOCOFing · synthetic chimeric fusion**

Upload this notebook to Kaggle, attach **ruizgara/socofing** and
**jessicali9530/lfw-dataset**, enable **GPU T4** and **Internet**, then Run All.
The input paths can be overridden in the configuration cell.
No old CSVs, checkpoints, or companion Python files are required.

This is a new experiment: biometric identities are disjoint across fitting,
model selection, calibration, and testing. One Real fingerprint is the
reference; its nine synthetic alterations are probes. Faces come from unrelated
people, so results describe **synthetic fusion and alteration robustness**, not
real same-person multimodal acquisition or presentation-attack resistance.
"Unseen" means excluded from task-specific fitting; VGGFace2 pretraining
identity overlap has not been audited.

Default execution trains a fingerprint encoder, evaluates a pretrained face
baseline, optionally fine-tunes the face encoder, selects models on validation,
calibrates on development, then evaluates the untouched test split. Three
prespecified chimeric mappings measure mapping sensitivity without retraining.
This is one training split/seed, not three independent training experiments.
This follow-up keeps the previous split for a controlled exploratory comparison.
It is not a new untouched population after inspection of the November results.
Selection now uses TAR at the target FAR, EER, then normalized tail separation.
Model-validation trials choose fusion weights; development trials calibrate
thresholds. An optional conservative threshold is reported beside the original
FAR-budget operating point, so any change in genuine acceptance stays visible.

Old metrics are not carried over. Correcting contamination can increase EER.
Save the executed notebook and download the final ZIP for your project report.


In [ ]:
import base64, zlib
OPTIMIZED_SOURCE_TEXT = zlib.decompress(base64.b64decode('...')).decode()
# Install the face package BEFORE importing torch. Preserve Kaggle's CUDA stack.
import importlib.metadata as metadata
import subprocess
import sys

FACENET_VERSION = "2.6.0"
try:
    installed_facenet = metadata.version("facenet-pytorch")
except metadata.PackageNotFoundError:
    installed_facenet = None
if installed_facenet != FACENET_VERSION:
    if "facenet_pytorch" in sys.modules:
        raise RuntimeError("Restart the kernel, then run the installation cell first.")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
        f"facenet-pytorch=={FACENET_VERSION}",
    ])
print("Using Kaggle's installed torch/torchvision/NumPy stack with facenet-pytorch", FACENET_VERSION)
# Only the InceptionResnetV1 encoder is used. Model smoke checks below exercise
# this combination; exact runtime versions are exported with the results.


## 1. Configuration and runtime

Set input directories only if automatic discovery is ambiguous. Keep the seed,
mapping list, primary condition, fusion objective, and FAR target fixed before
examining test results. Use a new RUN_NAME for a changed experiment. A default
timestamp prevents accidental checkpoint overwrites during a fresh Run All.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict, OrderedDict
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext
import copy
import hashlib
import json
import math
import os
import random
import re
import shutil
import time

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from tqdm.auto import tqdm

import torch

def check_torch_runtime(*, synchronize_cuda=False):
    """Detect incomplete/stale imports before expensive inventory or GPU work."""
    missing = [name for name in ("device", "Tensor", "tensor")
               if not callable(getattr(torch, name, None))]
    same_module = sys.modules.get("torch") is torch
    details = (f"torch.__file__={getattr(torch, '__file__', None)!r}; "
               f"torch.__version__={getattr(torch, '__version__', None)!r}; "
               f"missing/noncallable APIs={missing}; module_matches_sys_modules={same_module}")
    recovery = "Restart the Kaggle session, then Run All from the installation cell using Kaggle's installed torch/torchvision stack."
    if missing or not same_module:
        raise RuntimeError(f"The PyTorch import is incomplete or stale. {details}. {recovery}")
    if synchronize_cuda and device.type == "cuda":
        try:
            torch.cuda.synchronize(device)
        except AttributeError as error:
            raise RuntimeError(f"PyTorch's CUDA runtime failed its import check. {details}. {recovery}") from error

check_torch_runtime()
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision
from torchvision import models, transforms
from facenet_pytorch import InceptionResnetV1

CFG = {
    "SOCOFING_ROOT": "/kaggle/input/datasets/ruizgara/socofing/SOCOFing",  
    "LFW_IMAGES": "/kaggle/input/datasets/jessicali9530/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled",     
    "OUTPUT_ROOT": "/kaggle/working",
    "RUN_NAME": "biometric_v2_optimized_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
    "PROTOCOL_VERSION": "2026-11-09-selection-calibration-v2",
    "SPLIT_SEED": 20261030,
    "TRAIN_SEED": 42,
    "MAPPING_SEEDS": [42, 123, 2026],
    "N_MODEL_VAL": 45,
    "N_DEV": 52,
    "N_TEST": 106,
    "FACE_FIT_MIN_IMAGES": 10,
    "FACE_FIT_MAX_IMAGES": 20,
    "FINGER_IMAGE_SIZE": 112,
    "FACE_IMAGE_SIZE": 160,
    "FINGER_EPOCHS": 25,
    "FACE_EPOCHS": 15,
    "FINE_TUNE_FACE": True,
    "FINGER_LR": 3e-4,
    "FACE_LR": 2e-5,
    "PATIENCE": 6,
    "BATCH_SIZE": 64,
    "FACE_P": 16,
    "FACE_K": 4,
    "NUM_WORKERS": 0,  # Avoid the observed notebook multiprocessing cleanup errors.
    "DECODED_IMAGE_CACHE_MB": 512,
    "INVENTORY_IO_WORKERS": 4,
    "INVENTORY_CACHE_DIR": "/kaggle/working/biometric_inventory_cache_v2",
    "INVENTORY_CACHE_READ_DIR": "",  # Optional uploaded cache directory under /kaggle/input.
    "STRICT_INVENTORY_REHASH": True,  # First-run audit default; False permits metadata-validated reuse.
    "BOOTSTRAP_NOTE": "Conditional identity-node intervals; zero-error bounds are not population guarantees",
    "AMP": True,
    "MAX_CONSECUTIVE_AMP_SKIPS": 20,
    "REQUIRE_CUDA": True,
    "TARGET_FAR": 1e-3,
    "MODEL_SELECTION_OBJECTIVE": "tar_at_target_far",  # alternative: eer
    "SELECTION_MIN_TAIL_GAIN": 1e-3,
    "FUSION_OBJECTIVE": "tar_at_target_far",  # alternative: eer
    "FUSION_TIEBREAK": "center",  # tail_gap is an explicit experiment, not the default
    "REPORT_CONSERVATIVE_THRESHOLD": True,
    "BOOTSTRAP_REPLICATES": 2000,
    "PRIMARY_CONDITION": "All",
}
CONDITIONS = [f"{tier}/{kind}" for tier in ("Easy", "Medium", "Hard")
              for kind in ("CR", "Obl", "Zcut")]
PARTITIONS = ("fit", "model_val", "dev", "test")
assert CFG["INVENTORY_IO_WORKERS"] >= 1
assert CFG["DECODED_IMAGE_CACHE_MB"] >= 0 and CFG["NUM_WORKERS"] >= 0
assert len(set(CFG["MAPPING_SEEDS"])) == len(CFG["MAPPING_SEEDS"])
assert 0 <= CFG["TARGET_FAR"] < 1
assert CFG["FUSION_OBJECTIVE"] in {"eer", "tar_at_target_far"}
assert CFG["MODEL_SELECTION_OBJECTIVE"] in {"eer", "tar_at_target_far"}
assert CFG["FUSION_TIEBREAK"] in {"center", "tail_gap"}
assert CFG["SELECTION_MIN_TAIL_GAIN"] >= 0
assert CFG["MAX_CONSECUTIVE_AMP_SKIPS"] >= 1
assert CFG["PRIMARY_CONDITION"] == "All", "This version calibrates on all nine conditions."

OUT = Path(CFG["OUTPUT_ROOT"]) / CFG["RUN_NAME"]
OUT.mkdir(parents=True, exist_ok=False)
for folder in ("manifests", "checkpoints", "embeddings", "calibration", "results"):
    (OUT / folder).mkdir()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if CFG["REQUIRE_CUDA"] and device.type != "cuda":
    raise RuntimeError("Enable GPU T4 in Kaggle Settings. Set REQUIRE_CUDA=False only for a local smoke run.")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, indent=2, allow_nan=False) + "\n")

stage_timings = []

def record_stage(name, started, *, synchronize_cuda=False):
    # Inventory hashing and calibration are CPU/I/O work. Only GPU stages
    # need to wait for queued CUDA kernels before reading the wall clock.
    if synchronize_cuda and device.type == "cuda":
        check_torch_runtime(synchronize_cuda=True)
    stage_timings.append({"stage": name, "seconds": time.perf_counter() - started})
    pd.DataFrame(stage_timings).to_csv(OUT / "stage_timings.csv", index=False)

seed_everything(CFG["TRAIN_SEED"])
check_torch_runtime(synchronize_cuda=True)
environment = {name: metadata.version(name) for name in (
    "torch", "torchvision", "facenet-pytorch", "numpy", "pandas", "scipy", "scikit-learn", "Pillow")}
environment.update(python=sys.version, cuda=torch.version.cuda,
                   gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
write_json(OUT / "config.json", CFG)
write_json(OUT / "environment.json", environment)
(OUT / "installed_packages.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True))
print(json.dumps(environment, indent=2))
print("Run directory:", OUT)
print("Deterministic settings reduce variability; cross-version/GPU bitwise equality is not guaranteed.")


## 2. Rebuild and validate the image inventory

Select one complete, standard-sized finger family per subject. Each family must
have one Real image and all nine alteration conditions. Canonical pixel hashes
catch exact content duplicates even when file encodings differ. Exclusions are
saved. Hashing is not a perceptual near-duplicate audit.

Reading and hashing the images is a one-time CPU/I/O step. The resulting CSVs
contain explicit paths, biological identity keys, source families, and hashes.


In [ ]:
def discover_root(explicit, kind, search_root=Path("/kaggle/input")):
    if explicit:
        candidates = [Path(explicit)]
    else:
        leaf = "SOCOFing" if kind == "finger" else "lfw-deepfunneled"
        candidates = sorted(p for p in search_root.rglob(leaf) if p.is_dir())
    valid = []
    for path in candidates:
        if kind == "finger":
            okay = (path / "Real").is_dir() and (path / "Altered").is_dir()
        else:
            okay = any(child.is_dir() and any(child.glob("*.jpg")) for child in path.iterdir())
        if okay:
            valid.append(path.resolve())
    valid = sorted(set(valid))
    if len(valid) != 1:
        raise ValueError(f"Expected one {kind} image root; found {valid}. Set its explicit CFG path.")
    return valid[0]

def pixel_digest(path, mode):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image).convert(mode)
        image.load()
        size = image.size
        digest = hashlib.sha256(f"{mode}:{size}".encode() + image.tobytes()).hexdigest()
    return digest, size

def fingerprint_inventory(root, seed):
    pattern = re.compile(
        r"^(\d+)__([MF])_(Left|Right)_(index|middle|ring|little|thumb)_finger"
        r"(?:_(CR|Obl|Zcut))?\.bmp$", re.IGNORECASE)
    rows, excluded = [], []
    for path in inventory_file_lists.get(("finger", str(root.resolve()))) or sorted(root.rglob("*")):
        if not path.is_file() or path.suffix.lower() != ".bmp":
            continue
        match = pattern.fullmatch(path.name)
        if not match:
            excluded.append({"path": str(path), "reason": "unrecognized_filename"})
            continue
        subject, gender, hand, finger, alteration = match.groups()
        subject, hand, finger = str(int(subject)), hand.title(), finger.lower()
        if path.parent.name == "Real" and alteration is None:
            condition = "Real"
        elif path.parent.name in {"Altered-Easy", "Altered-Medium", "Altered-Hard"} and alteration:
            kind = {"cr": "CR", "obl": "Obl", "zcut": "Zcut"}[alteration.lower()]
            condition = path.parent.name.replace("Altered-", "") + "/" + kind
        else:
            excluded.append({"path": str(path), "reason": "folder_filename_mismatch"})
            continue
        family = f"{subject}:{hand}:{finger}"
        rows.append({"identity": subject, "gender": gender.upper(), "hand": hand, "finger": finger,
                     "source_family": family, "condition": condition, "filepath": str(path),
                     "sample_id": "finger:" + family + ":" + condition, "modality": "finger"})
    raw = pd.DataFrame(rows)
    if raw.empty:
        raise ValueError(f"No SOCOFing images found under {root}")
    expected = set(CONDITIONS + ["Real"])
    families = {key: group for key, group in raw.groupby("source_family", sort=True)
                if len(group) == 10 and set(group.condition) == expected}
    rng = np.random.default_rng(seed)
    jobs = []
    for subject in sorted(raw.identity.unique(), key=int):
        keys = sorted(key for key in families if key.split(":")[0] == subject)
        jobs.append((subject, rng.permutation(keys)))
    def inspect_subject(job):
        subject, keys = job
        failures = []
        for key in keys:
            group = families[key].copy()
            try:
                content = [pixel_digest(path, "L") for path in group.filepath]
            except (OSError, ValueError) as error:
                failures.append({"path": key, "reason": "unreadable_family:" + str(error)})
                continue
            if any(size != (96, 103) for _, size in content):
                failures.append({"path": key, "reason": "nonstandard_dimensions"})
                continue
            group["pixel_sha256"] = [digest for digest, _ in content]
            reference_hash = group.loc[group.condition == "Real", "pixel_sha256"].iloc[0]
            if (group.loc[group.condition != "Real", "pixel_sha256"] == reference_hash).any():
                failures.append({"path": key, "reason": "probe_equals_reference_pixels"})
                continue
            group["role"] = np.where(group.condition == "Real", "reference", "probe")
            return group, failures
        failures.append({"path": "subject:" + subject, "reason": "no_eligible_family"})
        return None, failures
    selected = []
    with ThreadPoolExecutor(max_workers=CFG["INVENTORY_IO_WORKERS"]) as pool:
        for group, failures in tqdm(pool.map(inspect_subject, jobs), total=len(jobs), desc="Hash fingerprint families"):
            excluded.extend(failures)
            if group is not None:
                selected.append(group)
    if not selected:
        raise ValueError("No complete standard-sized fingerprint families survived validation.")
    frame = pd.concat(selected, ignore_index=True)
    return frame, pd.DataFrame(excluded, columns=["path", "reason"])

def face_inventory(root):
    def inspect_person(person):
        records, excluded, seen = [], [], set()
        for path in sorted(person.glob("*.jpg")):
            try:
                digest, size = pixel_digest(path, "RGB")
            except (OSError, ValueError) as error:
                excluded.append({"path": str(path), "reason": "unreadable:" + str(error)})
                continue
            if digest in seen:
                excluded.append({"path": str(path), "reason": "duplicate_pixels_within_identity"})
                continue
            seen.add(digest)
            records.append({"identity": person.name, "filepath": str(path), "pixel_sha256": digest,
                            "sample_id": "face:" + person.name + ":" + path.name,
                            "source_family": "face:" + digest, "modality": "face",
                            "width": size[0], "height": size[1]})
        return records, excluded
    people = sorted(person for person in root.iterdir() if person.is_dir())
    records, excluded = [], []
    with ThreadPoolExecutor(max_workers=CFG["INVENTORY_IO_WORKERS"]) as pool:
        for person_records, person_exclusions in tqdm(pool.map(inspect_person, people), total=len(people), desc="Hash LFW images"):
            records.extend(person_records)
            excluded.extend(person_exclusions)
    frame = pd.DataFrame(records)
    if frame.empty:
        raise ValueError(f"No face images found under {root}")
    counts = frame.groupby("identity").size()
    return frame[frame.identity.isin(counts[counts >= 2].index)].reset_index(drop=True), pd.DataFrame(
        excluded, columns=["path", "reason"])

def assert_unambiguous_content(frame):
    ambiguous = frame.groupby("pixel_sha256").identity.nunique()
    bad = ambiguous[ambiguous > 1].index
    if len(bad):
        examples = frame[frame.pixel_sha256.isin(bad)][["identity", "filepath"]].head(12)
        raise ValueError("Identical content has different identity labels; resolve before splitting:\n" + examples.to_string(index=False))
    assert frame.sample_id.is_unique
    assert frame.filepath.is_unique


INVENTORY_CACHE_VERSION = "v2-20261111-preserved-family-selection"
inventory_cache_events = []
inventory_file_lists = {}

def cached_inventory(root, modality, builder):
    """Reuse pixel hashes only for an unchanged file-metadata inventory.

    Intended for immutable Kaggle dataset versions. STRICT_INVENTORY_REHASH
    disables reuse; unchanged size/mtime is not a new byte-level audit.
    Paths in the cache are relative so a dataset can move between mounts.
    """
    root = Path(root).resolve()
    suffix = ".bmp" if modality == "finger" else ".jpg"
    paths = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() == suffix)
    inventory_file_lists[(modality, str(root))] = paths
    listing = hashlib.sha256()
    for path in paths:
        info = path.stat()
        listing.update(json.dumps([path.relative_to(root).as_posix(), info.st_size, info.st_mtime_ns]).encode())
        listing.update(b"\n")
    signature = {"version": INVENTORY_CACHE_VERSION, "modality": modality,
                 "seed": CFG["SPLIT_SEED"], "file_count": len(paths),
                 "listing_sha256": listing.hexdigest()}
    cache_name = modality + "_inventory.json"
    destinations = [Path(CFG["INVENTORY_CACHE_DIR"])]
    if CFG["INVENTORY_CACHE_READ_DIR"]:
        destinations.insert(0, Path(CFG["INVENTORY_CACHE_READ_DIR"]))
    if not CFG["STRICT_INVENTORY_REHASH"]:
        for directory in destinations:
            path = directory / cache_name
            if not path.is_file():
                continue
            try:
                payload = json.loads(path.read_text())
                if payload["signature"] != signature:
                    continue
                frame = pd.DataFrame(payload["records"])
                for relative in frame.filepath:
                    resolved = (root / relative).resolve()
                    if not resolved.is_relative_to(root):
                        raise ValueError("Inventory cache path escapes its dataset root")
                frame["filepath"] = frame.filepath.map(lambda relative: str(root / relative))
                exclusions = pd.DataFrame(payload["exclusions"], columns=["path", "reason"])
                assert frame.sample_id.is_unique and frame.filepath.is_unique
            except (KeyError, ValueError, TypeError, AssertionError) as error:
                print(f"Ignoring invalid {modality} inventory cache: {error}")
                continue
            inventory_cache_events.append({**signature, "mode": "metadata_validated_cache",
                                           "cache_sha256": sha256_bytes(path.read_bytes())})
            print(f"Reused {modality} inventory; dataset listing/size/mtime match. No fresh pixel rehash.")
            return frame, exclusions
    frame, exclusions = builder(root)
    portable = frame.copy()
    portable["filepath"] = portable.filepath.map(lambda value: Path(value).relative_to(root).as_posix())
    directory = Path(CFG["INVENTORY_CACHE_DIR"])
    directory.mkdir(parents=True, exist_ok=True)
    destination = directory / cache_name
    payload = {"signature": signature, "records": portable.to_dict(orient="records"),
               "exclusions": exclusions.to_dict(orient="records")}
    temporary = destination.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(payload, allow_nan=False))
    temporary.replace(destination)
    inventory_cache_events.append({**signature, "mode": "fresh_pixel_hashes",
                                   "cache_sha256": sha256_bytes(destination.read_bytes())})
    return frame, exclusions

def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()

inventory_started = time.perf_counter()
SOCOFING_ROOT = discover_root(CFG["SOCOFING_ROOT"], "finger")
LFW_IMAGES = discover_root(CFG["LFW_IMAGES"], "face")
print("SOCOFing:", SOCOFING_ROOT, "\nLFW:", LFW_IMAGES)
finger_inventory, finger_exclusions = cached_inventory(SOCOFING_ROOT, "finger", lambda root: fingerprint_inventory(root, CFG["SPLIT_SEED"]))
face_inventory_df, face_exclusions = cached_inventory(LFW_IMAGES, "face", face_inventory)
for modality, frame, exclusions in (("finger", finger_inventory, finger_exclusions),
                                     ("face", face_inventory_df, face_exclusions)):
    assert_unambiguous_content(frame)
    frame.to_csv(OUT / "manifests" / f"{modality}_inventory.csv", index=False)
    exclusions.to_csv(OUT / "manifests" / f"{modality}_exclusions.csv", index=False)
    print(modality, "eligible identities:", frame.identity.nunique(), "images:", len(frame), "exclusions:", len(exclusions))

record_stage("inventory", inventory_started)


## 3. Split biological identities before making chimeric pairs

Fingerprint defaults reserve 45 identities for model selection, 52 for
calibration, and 106 for testing; the remaining identities fit the model.
Faces with at least ten unique images fit the face model. Separate identities
with two to nine images supply model validation, development, and testing.
This matches the available LFW cohort sizes but introduces a sampling
difference; document it in your report. Only two face images per evaluation
identity are selected, before training. The same face pair is reused across
the nine fingerprint conditions and must not be counted as nine independent
face acquisitions.


In [ ]:
def make_partition_map(identities, n_val, n_dev, n_test, seed):
    ids = np.array(sorted(set(map(str, identities))))
    required = n_val + n_dev + n_test
    if len(ids) <= required:
        raise ValueError(f"Need more than {required} identities; found {len(ids)}.")
    ids = np.random.default_rng(seed).permutation(ids)
    groups = {"model_val": ids[:n_val], "dev": ids[n_val:n_val+n_dev],
              "test": ids[n_val+n_dev:required], "fit": ids[required:]}
    return {identity: partition for partition, values in groups.items() for identity in values}

def assign_face_partitions(frame, n_val, n_dev, n_test, seed, min_fit_images, max_fit_images):
    counts = frame.groupby("identity").size()
    fit_ids = set(counts[counts >= min_fit_images].index)
    evaluation_ids = sorted(counts[(counts >= 2) & (counts < min_fit_images)].index)
    required = n_val + n_dev + n_test
    if len(evaluation_ids) < required or not fit_ids:
        raise ValueError("Insufficient unique face identities for this protocol; inspect the inventory and configuration.")
    rng = np.random.default_rng(seed)
    chosen = rng.permutation(evaluation_ids)[:required]
    mapping = {identity: "fit" for identity in fit_ids}
    for partition, values in (("model_val", chosen[:n_val]), ("dev", chosen[n_val:n_val+n_dev]),
                               ("test", chosen[n_val+n_dev:])):
        mapping.update({identity: partition for identity in values})
    pieces = []
    for identity, group in frame.groupby("identity", sort=True):
        if identity not in mapping:
            continue
        partition = mapping[identity]
        number = min(len(group), max_fit_images) if partition == "fit" else 2
        selected = group.iloc[rng.permutation(len(group))[:number]].copy()
        selected["partition"] = partition
        selected["role"] = "fit" if partition == "fit" else ["reference", "probe"]
        selected["condition"] = "Face"
        pieces.append(selected)
    return pd.concat(pieces, ignore_index=True)

def assert_partition_integrity(frame):
    assert frame.partition.notna().all()
    assert set(frame.partition) == set(PARTITIONS)
    for key in ("identity", "source_family", "filepath", "pixel_sha256"):
        assert (frame.groupby(key).partition.nunique() == 1).all(), f"Cross-partition overlap: {key}"
    for partition in ("model_val", "dev", "test"):
        subset = frame[frame.partition == partition]
        for identity, group in subset.groupby("identity"):
            reference = group[group.role == "reference"]
            probes = group[group.role == "probe"]
            assert len(reference) == 1 and len(probes) >= 1, (partition, identity)
            assert not set(reference.pixel_sha256) & set(probes.pixel_sha256), "Self-comparison"

finger_df = finger_inventory.copy()
finger_map = make_partition_map(finger_df.identity, CFG["N_MODEL_VAL"], CFG["N_DEV"],
                                CFG["N_TEST"], CFG["SPLIT_SEED"])
finger_df["partition"] = finger_df.identity.map(finger_map)
finger_df.loc[finger_df.partition == "fit", "role"] = "fit"
face_df = assign_face_partitions(face_inventory_df, CFG["N_MODEL_VAL"], CFG["N_DEV"], CFG["N_TEST"],
                                 CFG["SPLIT_SEED"] + 1, CFG["FACE_FIT_MIN_IMAGES"], CFG["FACE_FIT_MAX_IMAGES"])
assert face_df[face_df.partition == "fit"].identity.nunique() >= CFG["FACE_P"]
assert finger_df[finger_df.partition == "fit"].identity.nunique() >= 2
for modality, frame in (("face", face_df), ("finger", finger_df)):
    assert_partition_integrity(frame)
    frame.to_csv(OUT / "manifests" / f"{modality}_split.csv", index=False)
    print(modality)
    display(frame.groupby("partition").agg(identities=("identity", "nunique"), images=("sample_id", "size")))
print("Identity, source-family, path, and exact-content partition checks passed.")
display(pd.crosstab(finger_df.partition, finger_df.condition))
print("Fingerprint subject gender counts:")
display(pd.crosstab(finger_df.drop_duplicates("identity").partition,
                    finger_df.drop_duplicates("identity").gender))


## 4. Shared preprocessing, sampler, and encoders

Every face path uses fixed standardization `(pixel - 127.5) / 128` after
resizing to 160×160. Fingerprints are grayscale and normalized to [-1, 1].
Evaluation has no random transforms. LFW deep-funneled images are already
aligned; this notebook does not add a different face detector at inference.

The PK sampler visits every training identity each round and cycles through
each identity's images. Early face BatchNorm statistics remain frozen when
their parameters are frozen. FP16 autocast is used on T4; normalization,
triplet distances, and ArcFace margin/loss calculations use float32.


In [ ]:
def face_standardize(tensor):
    return (tensor * 255.0 - 127.5) / 128.0

check_torch_runtime(synchronize_cuda=True)

def make_transform(modality, training=False):
    size = CFG["FACE_IMAGE_SIZE"] if modality == "face" else CFG["FINGER_IMAGE_SIZE"]
    operations = [transforms.Resize((size, size), antialias=True)]
    if training and modality == "face":
        operations += [transforms.RandomHorizontalFlip(), transforms.RandomRotation(7),
                       transforms.ColorJitter(brightness=0.15, contrast=0.15)]
    elif training:
        operations += [transforms.RandomRotation(10)]
    operations += [transforms.ToTensor()]
    operations += ([transforms.Lambda(face_standardize)] if modality == "face"
                   else [transforms.Normalize([0.5], [0.5])])
    return transforms.Compose(operations)

# Bounded shared cache in the main process. Resize occurs before augmentation,
# exactly as in v2. Store uint8 pixels; stochastic transforms run on every access.
decoded_image_cache = OrderedDict()
decoded_image_cache_bytes = 0

class ImageTable(Dataset):
    def __init__(self, frame, modality, training=False, label_map=None):
        self.frame = frame.reset_index(drop=True).copy()
        self.modality = modality
        pipeline = make_transform(modality, training)
        self.resize = pipeline.transforms[0]
        self.transform = transforms.Compose(pipeline.transforms[1:])
        self.size = CFG["FACE_IMAGE_SIZE"] if modality == "face" else CFG["FINGER_IMAGE_SIZE"]
        self.paths = self.frame.filepath.tolist()
        self.hashes = self.frame.pixel_sha256.tolist()
        self.label_map = label_map or {key: i for i, key in enumerate(sorted(self.frame.identity.unique()))}
        self.labels = self.frame.identity.map(self.label_map).to_numpy()
        assert not pd.isna(self.labels).any()
        self.label_to_indices = {int(label): np.flatnonzero(self.labels == label).tolist()
                                 for label in np.unique(self.labels)}

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        global decoded_image_cache_bytes
        key = (self.paths[index], self.hashes[index], self.modality, self.size)
        # Worker processes would each own a copy of this cache. Disable it when
        # workers are enabled instead of multiplying memory use unpredictably.
        budget = int(CFG["DECODED_IMAGE_CACHE_MB"] * 1024**2) if CFG["NUM_WORKERS"] == 0 else 0
        pixels = decoded_image_cache.get(key) if budget else None
        if pixels is None:
            with Image.open(self.paths[index]) as image:
                image = ImageOps.exif_transpose(image).convert("RGB" if self.modality == "face" else "L")
                pixels = np.array(self.resize(image), copy=True)
            if budget and pixels.nbytes <= budget:
                while decoded_image_cache and decoded_image_cache_bytes + pixels.nbytes > budget:
                    _, previous = decoded_image_cache.popitem(last=False)
                    decoded_image_cache_bytes -= previous.nbytes
                decoded_image_cache[key] = pixels
                decoded_image_cache_bytes += pixels.nbytes
        elif budget:
            decoded_image_cache.move_to_end(key)
        # Copy prevents any downstream transform from changing the cached image.
        tensor = self.transform(Image.fromarray(pixels.copy()))
        return tensor, int(self.labels[index])

class CoveragePKSampler(Sampler):
    def __init__(self, dataset, p=16, k=4, seed=42):
        self.groups = dataset.label_to_indices
        self.labels = sorted(self.groups)
        self.p, self.k, self.seed, self.epoch = p, k, seed, 0
        assert len(self.labels) >= p and p >= 2 and k >= 2
        assert min(map(len, self.groups.values())) >= k
        self.rounds = math.ceil(max(map(len, self.groups.values())) / k)

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return math.ceil(len(self.labels) / self.p) * self.rounds

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch)
        decks = {label: rng.sample(indices, len(indices)) for label, indices in self.groups.items()}
        for _ in range(self.rounds):
            labels = rng.sample(self.labels, len(self.labels))
            for start in range(0, len(labels), self.p):
                chosen_labels = labels[start:start+self.p]
                if len(chosen_labels) < self.p:
                    chosen_labels += rng.sample([label for label in self.labels if label not in chosen_labels],
                                                 self.p - len(chosen_labels))
                batch = []
                for label in chosen_labels:
                    chosen = []
                    while len(chosen) < self.k:
                        if not decks[label]:
                            decks[label] = rng.sample(self.groups[label], len(self.groups[label]))
                        candidate = decks[label].pop()
                        if candidate not in chosen:
                            chosen.append(candidate)
                    batch.extend(chosen)
                yield batch

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def make_loader(frame, modality, training=False, pk=False):
    dataset = ImageTable(frame, modality, training)
    kwargs = dict(num_workers=CFG["NUM_WORKERS"], pin_memory=device.type == "cuda",
                  worker_init_fn=seed_worker,
                  generator=torch.Generator().manual_seed(CFG["TRAIN_SEED"]),
                  persistent_workers=False)
    if pk:
        return DataLoader(dataset, batch_sampler=CoveragePKSampler(
            dataset, CFG["FACE_P"], CFG["FACE_K"], CFG["TRAIN_SEED"]), **kwargs)
    return DataLoader(dataset, batch_size=CFG["BATCH_SIZE"], shuffle=training,
                      drop_last=training, **kwargs)

class FaceEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = InceptionResnetV1(pretrained="vggface2" if pretrained else None)
        # The pretrained constructor adds an unused identity classifier. Remove
        # its weights so encoder-only checkpoints load strictly without a download.
        self.backbone.logits = nn.Identity()

    def enable_fine_tuning(self):
        for name, parameter in self.backbone.named_parameters():
            parameter.requires_grad = name.startswith(("repeat_3.", "block8.", "last_linear.", "last_bn."))

    def train(self, mode=True):
        super().train(mode)
        if mode:
            for module in self.backbone.modules():
                if isinstance(module, nn.modules.batchnorm._BatchNorm):
                    if not any(parameter.requires_grad for parameter in module.parameters()):
                        module.eval()
        return self

    def forward(self, images):
        return F.normalize(self.backbone(images).float(), dim=1)

class FingerEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        previous = self.backbone.conv1
        self.backbone.conv1 = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            self.backbone.conv1.weight.copy_(previous.weight.mean(dim=1, keepdim=True))
        self.backbone.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(512, 512), nn.BatchNorm1d(512))

    def forward(self, images):
        return F.normalize(self.backbone(images).float(), dim=1)

class ArcFaceHead(nn.Module):
    def __init__(self, classes, embedding_dim=512, margin=0.5, scale=30.0):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)
        self.margin, self.scale = margin, scale

    def forward(self, embeddings, labels):
        cosine = F.linear(F.normalize(embeddings.float()), F.normalize(self.weight.float()))
        cosine = cosine.clamp(-1 + 1e-7, 1 - 1e-7)
        sine = torch.sqrt((1 - cosine.square()).clamp_min(1e-7))
        phi = cosine * math.cos(self.margin) - sine * math.sin(self.margin)
        phi = torch.where(cosine > math.cos(math.pi - self.margin), phi,
                          cosine - math.sin(math.pi - self.margin) * self.margin)
        one_hot = F.one_hot(labels, num_classes=self.weight.shape[0]).to(cosine.dtype)
        return self.scale * (one_hot * phi + (1 - one_hot) * cosine)

def triplet_loss(embeddings, labels, margin=0.2):
    distances = torch.cdist(embeddings.float(), embeddings.float())
    same = labels[:, None].eq(labels[None, :])
    positive = same & ~torch.eye(len(labels), device=labels.device, dtype=torch.bool)
    negative = ~same
    assert positive.any(dim=1).all() and negative.any(dim=1).all()
    hard_positive = distances.masked_fill(~positive, -torch.inf).max(dim=1).values
    hard_negative = distances.masked_fill(~negative, torch.inf).min(dim=1).values
    return F.relu(hard_positive - hard_negative + margin).mean()

def autocast_context():
    if device.type == "cuda" and CFG["AMP"]:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()

def grad_scaler():
    enabled = device.type == "cuda" and CFG["AMP"]
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)

def backward_and_step(loss, parameters, optimizer, scaler):
    """Clip finite gradients; let an enabled GradScaler skip overflowed updates.

    Returns True only when the optimizer update was applied. This helper is for
    the single-optimizer, dense-gradient training loops in this notebook.
    """
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    gradients = [parameter.grad for parameter in parameters if parameter.grad is not None]
    if not gradients:
        raise RuntimeError("No gradients were produced for the training parameters.")
    # One device-to-host decision, rather than one synchronization per parameter.
    finite = bool(torch.stack([torch.isfinite(gradient).all() for gradient in gradients]).all().item())
    if finite:
        # Keep the strict check: even finite elements can overflow a norm
        # reduction. GradScaler would NOT automatically detect that case.
        nn.utils.clip_grad_norm_(parameters, max_norm=5.0, error_if_nonfinite=True)
    elif not scaler.is_enabled():
        raise FloatingPointError("Nonfinite gradients with AMP disabled; inspect the loss, inputs and model.")
    # unscale_ already recorded nonfinite gradients. On overflow, step skips
    # optimizer.step(), and update lowers the scale for the next batch.
    scaler.step(optimizer)
    scaler.update()
    if not finite:
        optimizer.zero_grad(set_to_none=True)
    return finite

@torch.no_grad()
def extract_embeddings(model, frame, modality):
    model.eval()
    loader = make_loader(frame, modality, training=False)
    outputs = []
    for images, _ in loader:
        with autocast_context():
            embeddings = model(images.to(device, non_blocking=True))
        outputs.append(F.normalize(embeddings.float(), dim=1).cpu().numpy())
    embeddings = np.concatenate(outputs)
    assert len(embeddings) == len(frame)
    assert np.isfinite(embeddings).all()
    assert np.allclose(np.linalg.norm(embeddings, axis=1), 1, atol=1e-4)
    return embeddings


In [ ]:
# Fail early if this Kaggle dependency combination cannot run these architectures.
# No pretrained download is needed for these forward/backward smoke checks.
for model_class, channels, size in ((FingerEncoder, 1, CFG["FINGER_IMAGE_SIZE"]),
                                     (FaceEncoder, 3, CFG["FACE_IMAGE_SIZE"])):
    smoke = model_class(pretrained=False).to(device).train()
    with autocast_context():
        outputs = smoke(torch.randn(2, channels, size, size, device=device))
    assert outputs.shape == (2, 512) and torch.isfinite(outputs).all()
    outputs[:, 0].sum().backward()
    assert all(torch.isfinite(p.grad).all() for p in smoke.parameters() if p.grad is not None)
    del smoke, outputs
if device.type == "cuda":
    torch.cuda.empty_cache()
seed_everything(CFG["TRAIN_SEED"])
print("Encoder forward/backward smoke checks passed.")


## 5. Verification metrics and model-selection validation

Distances always run **probe → reference**. EER is a descriptive interpolated
ROC summary, with ties handled as a group. No EER threshold is advertised as a
deployment threshold. The separate FAR-target threshold permits at most the
allowed number of development false accepts, including tied scores.
Validation also reports the actual FAR/TAR counts and a continuous tail gap:
(impostor FAR-quantile - genuine 99th percentile) / impostor standard deviation.
A larger gap means more separation near the operating region. This resolves
saturated EER/TAR ties; it is a diagnostic, not a population FAR guarantee.


In [ ]:
def cross_distances(probes, references):
    probes = np.asarray(probes, dtype=np.float32)
    references = np.asarray(references, dtype=np.float32)
    for array in (probes, references):
        assert array.ndim == 2 and np.isfinite(array).all()
        assert np.allclose(np.linalg.norm(array, axis=1), 1.0, atol=1e-4)
    return np.sqrt(np.clip(2.0 - 2.0 * (probes @ references.T), 0.0, 4.0))

def eer_from_curve(fpr, tpr):
    difference = fpr - (1.0 - tpr)
    right = int(np.flatnonzero(difference >= 0)[0])
    if right == 0:
        return float(fpr[0])
    left = right - 1
    denominator = difference[right] - difference[left]
    alpha = 0.0 if denominator == 0 else -difference[left] / denominator
    return float(fpr[left] + alpha * (fpr[right] - fpr[left]))

def roc_summary(distances, genuine):
    values = np.asarray(distances, dtype=np.float64)
    labels = np.asarray(genuine, dtype=bool)
    assert values.ndim == labels.ndim == 1 and len(values) == len(labels)
    assert np.isfinite(values).all() and labels.any() and (~labels).any()
    fpr, tpr, _ = roc_curve(labels.astype(int), -values, drop_intermediate=False)
    return {"eer": eer_from_curve(fpr, tpr), "auc": float(auc(fpr, tpr))}

def threshold_for_far(impostor_distances, far_target):
    scores = np.sort(np.asarray(impostor_distances, dtype=np.float64))
    assert scores.size and np.isfinite(scores).all() and 0 <= far_target < 1
    allowed = int(math.floor(far_target * len(scores) + 1e-12))
    # Strictly below the first disallowed score. If it belongs to a tied group,
    # exclude that entire group; <= threshold can never exceed the FAR budget.
    threshold = float(np.nextafter(scores[allowed], -np.inf))
    assert np.count_nonzero(scores <= threshold) <= allowed
    return threshold

def operating_metrics(distances, genuine, threshold):
    # Keep the nextafter boundary in float64, even for float32 embeddings.
    values, genuine = np.asarray(distances, dtype=np.float64), np.asarray(genuine, dtype=bool)
    assert values.ndim == genuine.ndim == 1 and len(values) == len(genuine)
    assert np.isfinite(values).all() and genuine.any() and (~genuine).any()
    accepted = values <= threshold
    tar = float(accepted[genuine].mean())
    return {"tar": tar, "frr": 1.0 - tar, "far": float(accepted[~genuine].mean()),
            "genuine_trials": int(genuine.sum()), "impostor_trials": int((~genuine).sum()),
            "false_accepts": int(accepted[~genuine].sum()),
            "false_rejects": int((~accepted[genuine]).sum())}

def score_diagnostics(distances, genuine, target_far):
    values = np.asarray(distances, dtype=np.float64)
    labels = np.asarray(genuine, dtype=bool)
    roc = roc_summary(values, labels)
    impostors, positives = values[~labels], values[labels]
    threshold = threshold_for_far(impostors, target_far)
    operating = operating_metrics(values, labels, threshold)
    impostor_tail = float(np.quantile(impostors, target_far, method="lower"))
    genuine_tail = float(np.quantile(positives, 0.99, method="higher"))
    scale = max(float(impostors.std()), 1e-8)
    return {**roc, **operating, "threshold": threshold, "target_far": float(target_far),
            "genuine_q99": genuine_tail, "genuine_max": float(positives.max()),
            "impostor_far_quantile": impostor_tail, "impostor_min": float(impostors.min()),
            "tail_gap": (impostor_tail - genuine_tail) / scale,
            "strict_gap": (float(impostors.min()) - float(positives.max())) / scale,
            "far_resolution": 1.0 / len(impostors)}

def selection_key(metrics, objective):
    assert objective in {"eer", "tar_at_target_far"}
    primary = (metrics["eer"], -metrics["tar"]) if objective == "eer" else (-metrics["tar"], metrics["eer"])
    return (*primary, -metrics["tail_gap"])

def selection_improves(candidate, incumbent, objective, tail_tolerance=1e-3):
    if incumbent is None:
        return True
    for new, old, tolerance in zip(selection_key(candidate, objective), selection_key(incumbent, objective),
                                    (1e-12, 1e-12, tail_tolerance)):
        if new < old - tolerance:
            return True
        if new > old + tolerance:
            return False
    return False  # Keep the earlier checkpoint on a complete tie.

def calibrated_threshold(distances, genuine, target_far, policy="far_budget"):
    values = np.asarray(distances, dtype=np.float64)
    labels = np.asarray(genuine, dtype=bool)
    assert values.ndim == labels.ndim == 1 and len(values) == len(labels)
    assert labels.any() and (~labels).any() and np.isfinite(values).all()
    assert policy in {"far_budget", "genuine_midpoint"}
    budget = threshold_for_far(values[~labels], target_far)
    if policy == "far_budget":
        return budget
    accepted_genuine = values[labels & (values <= budget)]
    if not accepted_genuine.size:
        return budget
    # Prespecified half-gap rule: retain every genuine accepted by the FAR-budget
    # threshold, but use only half its unused score gap as a generalization cushion.
    # It can change test TAR, so it is a separate reported operating point.
    anchor = float(accepted_genuine.max())
    threshold = anchor + 0.5 * (budget - anchor)
    assert anchor <= threshold <= budget
    return float(threshold)

def unimodal_validation(model, frame, modality):
    assert set(frame.partition) == {"model_val"}, "Model selection is restricted to model_val."
    embeddings = extract_embeddings(model, frame, modality)
    references = np.flatnonzero(frame.role.to_numpy() == "reference")
    probes = np.flatnonzero(frame.role.to_numpy() == "probe")
    distances = cross_distances(embeddings[probes], embeddings[references])
    labels = frame.identity.to_numpy()
    genuine = labels[probes, None] == labels[references][None, :]
    result = score_diagnostics(distances.ravel(), genuine.ravel(), CFG["TARGET_FAR"])
    result["rank1"] = float((labels[references][distances.argmin(axis=1)] == labels[probes]).mean())
    result["identities"] = int(len(references))
    # Keep all nine related alterations together; do not call them independent
    # captures. Condition-specific rates use the SAME pooled validation threshold.
    condition_rates = []
    for condition in sorted(frame.iloc[probes].condition.unique()):
        mask = frame.iloc[probes].condition.to_numpy() == condition
        condition_rates.append(operating_metrics(distances[mask].ravel(), genuine[mask].ravel(), result["threshold"]))
    result["worst_condition_tar"] = min(item["tar"] for item in condition_rates)
    result["worst_condition_far"] = max(item["far"] for item in condition_rates)
    return result

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def checkpoint_metadata(modality, fit_frame, validation, epoch):
    return {"modality": modality, "epoch": int(epoch), "validation": validation,
            "fit_identities": sorted(fit_frame.identity.unique().tolist()),
            "train_seed": CFG["TRAIN_SEED"], "split_seed": CFG["SPLIT_SEED"],
            "selection_objective": CFG["MODEL_SELECTION_OBJECTIVE"],
            "selection_tiebreak": "secondary EER/TAR, then normalized impostor-FAR / genuine-q99 tail gap",
            "selection_min_tail_gain": CFG["SELECTION_MIN_TAIL_GAIN"],
            "architecture": "InceptionResnetV1" if modality == "face" else "ResNet18_grayscale",
            "preprocessing": "(pixel-127.5)/128" if modality == "face" else "grayscale_(pixel/255-0.5)/0.5",
            "split_sha256": sha256_file(OUT / "manifests" / f"{modality}_split.csv")}

def save_encoder(model, path, information):
    state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    torch.save({"state_dict": state, "metadata": information}, path)

def load_encoder(model, path):
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["state_dict"], strict=True)
    return checkpoint["metadata"]


## 6. Train a fresh fingerprint encoder

All variants of a fitting subject remain in fitting. Model validation uses
different subjects and fixed Real references. Select on validation TAR at the
target FAR, then EER, then tail separation. Later epochs can improve separation
even when EER is already zero. Training loss never selects the checkpoint.
The old fingerprint checkpoint is intentionally not an input to this run.
AMP overflow skips a parameter update and lowers the gradient scale. Histories
record applied/skipped updates; persistent overflow or an entirely skipped
epoch stops training before checkpoint selection.


In [ ]:
def train_fingerprint(fit_frame, val_frame):
    seed_everything(CFG["TRAIN_SEED"])
    model = FingerEncoder(pretrained=True).to(device)
    head = ArcFaceHead(fit_frame.identity.nunique()).to(device)
    loader = make_loader(fit_frame, "finger", training=True)
    assert len(loader) > 0
    parameters = list(model.parameters()) + list(head.parameters())
    optimizer = torch.optim.AdamW(parameters, lr=CFG["FINGER_LR"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["FINGER_EPOCHS"])
    scaler = grad_scaler()
    best, stale, history = None, 0, []
    consecutive_skips = 0
    path = OUT / "checkpoints" / "finger_resnet18_selected.pt"
    for epoch in range(CFG["FINGER_EPOCHS"]):
        started = time.perf_counter()
        model.train()
        head.train()
        losses = []
        optimizer_steps, skipped_steps = 0, 0
        for images, labels in tqdm(loader, desc=f"Fingerprint epoch {epoch+1}", leave=False):
            images, labels = images.to(device, non_blocking=True), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            with autocast_context():
                embeddings = model(images)
            loss = F.cross_entropy(head(embeddings.float(), labels), labels)
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite fingerprint loss.")
            applied = backward_and_step(loss, parameters, optimizer, scaler)
            if not applied:
                skipped_steps += 1
                consecutive_skips += 1
                print(f"Fingerprint AMP: skipped unsafe update; gradient scale now {scaler.get_scale():g}")
                if consecutive_skips >= CFG.get("MAX_CONSECUTIVE_AMP_SKIPS", 20):
                    raise FloatingPointError("Repeated fingerprint AMP overflow. Restart this training stage with AMP=False and record the change.")
                continue
            consecutive_skips = 0
            optimizer_steps += 1
            losses.append(float(loss.detach().cpu()))
        if optimizer_steps == 0:
            raise FloatingPointError("No fingerprint updates succeeded this epoch; no validation/checkpoint was saved. Try AMP=False.")
        validation = unimodal_validation(model, val_frame, "finger")
        improved = selection_improves(validation, best, CFG["MODEL_SELECTION_OBJECTIVE"],
                                      CFG["SELECTION_MIN_TAIL_GAIN"])
        history.append({"epoch": epoch+1, "loss": float(np.mean(losses)), **validation,
                        "selected_at_epoch": improved,
                        "optimizer_steps": optimizer_steps, "amp_skipped_steps": skipped_steps,
                        "amp_scale": scaler.get_scale(),
                        "seconds": time.perf_counter()-started, "lr": optimizer.param_groups[0]["lr"]})
        print(f"Fingerprint {epoch+1}: loss={np.mean(losses):.4f}, val EER={validation['eer']:.6%}, "
              f"TAR={validation['tar']:.4%}, FAR={validation['far']:.6%} "
              f"({validation['false_accepts']}/{validation['impostor_trials']}), "
              f"tail gap={validation['tail_gap']:.4f}, selected={improved}")
        if improved:
            best, stale = validation.copy(), 0
            save_encoder(model, path, checkpoint_metadata("finger", fit_frame, validation, epoch+1))
        else:
            stale += 1
        scheduler.step()
        pd.DataFrame(history).to_csv(OUT / "results" / "finger_training.csv", index=False)
        if stale >= CFG["PATIENCE"]:
            break
    information = load_encoder(model, path)
    del head, optimizer, scaler
    return model.eval(), information

finger_fit = finger_df[finger_df.partition == "fit"].reset_index(drop=True)
finger_val = finger_df[finger_df.partition == "model_val"].reset_index(drop=True)
fingerprint_training_started = time.perf_counter()
finger_model, finger_selection = train_fingerprint(finger_fit, finger_val)
write_json(OUT / "results" / "finger_selection.json", finger_selection)
record_stage("fingerprint_training", fingerprint_training_started, synchronize_cuda=True)
finger_model.cpu()
if device.type == "cuda":
    torch.cuda.empty_cache()


## 7. Face baseline and optional fine-tuning

Start from VGGFace2-pretrained InceptionResnetV1 with its native normalized
embedding. Validation includes subjects excluded from fine-tuning. Keep the
pretrained model if fine-tuning does not improve the declared validation rule.
Save the best adapted model separately even if the pretrained baseline wins.
Both pretrained-face and selected-face fusion are calibrated and reported.
The same guarded AMP update used for fingerprints also protects face training.


In [ ]:
def train_face(fit_frame, val_frame):
    seed_everything(CFG["TRAIN_SEED"])
    model = FaceEncoder(pretrained=True).to(device)
    initial_validation = unimodal_validation(model, val_frame, "face")
    baseline_path = OUT / "checkpoints" / "face_inceptionresnetv1_pretrained.pt"
    selected_path = OUT / "checkpoints" / "face_inceptionresnetv1_selected.pt"
    adapted_path = OUT / "checkpoints" / "face_inceptionresnetv1_adapted.pt"
    info = checkpoint_metadata("face", fit_frame, initial_validation, 0)
    info["fit_identities"] = []  # No task-specific fitting for the baseline.
    info["selection"] = "pretrained_baseline"
    save_encoder(model, baseline_path, info)
    save_encoder(model, selected_path, info)
    print(f"Pretrained face validation EER: {initial_validation['eer']:.3%}")
    history = [{"epoch": 0, "loss": None, **initial_validation, "seconds": 0.0}]
    if CFG["FINE_TUNE_FACE"]:
        model.enable_fine_tuning()
        loader = make_loader(fit_frame, "face", training=True, pk=True)
        parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
        optimizer = torch.optim.AdamW(parameters, lr=CFG["FACE_LR"], weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["FACE_EPOCHS"])
        scaler = grad_scaler()
        best, adapted_best, stale = initial_validation.copy(), None, 0
        consecutive_skips = 0
        for epoch in range(CFG["FACE_EPOCHS"]):
            started = time.perf_counter()
            loader.batch_sampler.set_epoch(epoch)
            model.train()
            losses = []
            optimizer_steps, skipped_steps = 0, 0
            for images, labels in tqdm(loader, desc=f"Face epoch {epoch+1}", leave=False):
                images, labels = images.to(device, non_blocking=True), labels.to(device)
                optimizer.zero_grad(set_to_none=True)
                with autocast_context():
                    embeddings = model(images)
                loss = triplet_loss(embeddings.float(), labels)
                if not torch.isfinite(loss):
                    raise FloatingPointError("Nonfinite face loss.")
                applied = backward_and_step(loss, parameters, optimizer, scaler)
                if not applied:
                    skipped_steps += 1
                    consecutive_skips += 1
                    print(f"Face AMP: skipped unsafe update; gradient scale now {scaler.get_scale():g}")
                    if consecutive_skips >= CFG.get("MAX_CONSECUTIVE_AMP_SKIPS", 20):
                        raise FloatingPointError("Repeated face AMP overflow. Restart this training stage with AMP=False and record the change.")
                    continue
                consecutive_skips = 0
                optimizer_steps += 1
                losses.append(float(loss.detach().cpu()))
            if optimizer_steps == 0:
                raise FloatingPointError("No face updates succeeded this epoch; no validation/checkpoint was saved. Try AMP=False.")
            validation = unimodal_validation(model, val_frame, "face")
            improved = selection_improves(validation, best, CFG["MODEL_SELECTION_OBJECTIVE"],
                                          CFG["SELECTION_MIN_TAIL_GAIN"])
            adapted_improved = selection_improves(validation, adapted_best, CFG["MODEL_SELECTION_OBJECTIVE"],
                                                  CFG["SELECTION_MIN_TAIL_GAIN"])
            history.append({"epoch": epoch+1, "loss": float(np.mean(losses)), **validation,
                            "selected_at_epoch": improved, "best_adapted_at_epoch": adapted_improved,
                            "optimizer_steps": optimizer_steps, "amp_skipped_steps": skipped_steps,
                            "amp_scale": scaler.get_scale(),
                            "seconds": time.perf_counter()-started, "lr": optimizer.param_groups[0]["lr"]})
            print(f"Face {epoch+1}: loss={np.mean(losses):.4f}, val EER={validation['eer']:.6%}, "
                  f"TAR={validation['tar']:.4%}, FAR={validation['far']:.6%} "
                  f"({validation['false_accepts']}/{validation['impostor_trials']}), "
                  f"tail gap={validation['tail_gap']:.4f}, selected={improved}")
            info = checkpoint_metadata("face", fit_frame, validation, epoch+1)
            info["selection"] = "fine_tuned"
            if adapted_improved:
                adapted_best = validation.copy()
                save_encoder(model, adapted_path, info)
                write_json(OUT / "results" / "face_adapted_selection.json", info)
            if improved:
                best, stale = validation.copy(), 0
                save_encoder(model, selected_path, info)
            else:
                stale += 1
            scheduler.step()
            pd.DataFrame(history).to_csv(OUT / "results" / "face_training.csv", index=False)
            if stale >= CFG["PATIENCE"]:
                break
        del optimizer, scaler
    pd.DataFrame(history).to_csv(OUT / "results" / "face_training.csv", index=False)
    information = load_encoder(model, selected_path)
    return model.eval(), information

face_fit = face_df[face_df.partition == "fit"].reset_index(drop=True)
face_val = face_df[face_df.partition == "model_val"].reset_index(drop=True)
face_training_started = time.perf_counter()
face_model, face_selection = train_face(face_fit, face_val)
write_json(OUT / "results" / "face_selection.json", face_selection)
record_stage("face_training", face_training_started, synchronize_cuda=True)
print("Selected face model:", face_selection["selection"], "epoch", face_selection["epoch"])
face_model.cpu()
if device.type == "cuda":
    torch.cuda.empty_cache()


## 8. Extract selection/development embeddings and create explicit trials

The model-validation partition selects fusion weights; development calibrates
thresholds. These stages never access test images or scores.
Embedding caches are keyed by the saved image manifest order. Each trial saves
its component sample IDs, preserving correspondence across all methods.
A reference-to-probe trial has one label shared by face and fingerprint.


In [ ]:
evaluation_encoder_cache = {}

def cache_partition_embeddings(partition):
    assert partition in {"model_val", "dev", "test"}
    face_frame = face_df[face_df.partition == partition].reset_index(drop=True)
    finger_frame = finger_df[finger_df.partition == partition].reset_index(drop=True)
    result = {}
    for name, modality, frame, model_class, checkpoint in (
        ("face_pretrained", "face", face_frame, FaceEncoder, "face_inceptionresnetv1_pretrained.pt"),
        ("face", "face", face_frame, FaceEncoder, "face_inceptionresnetv1_selected.pt"),
        ("finger", "finger", finger_frame, FingerEncoder, "finger_resnet18_selected.pt"),
    ):
        if checkpoint not in evaluation_encoder_cache:
            model = model_class(pretrained=False)
            load_encoder(model, OUT / "checkpoints" / checkpoint)
            evaluation_encoder_cache[checkpoint] = model.cpu().eval()
        model = evaluation_encoder_cache[checkpoint].to(device)
        embeddings = extract_embeddings(model, frame, modality)
        np.savez_compressed(OUT / "embeddings" / f"{partition}_{name}.npz",
                            sample_ids=frame.sample_id.to_numpy(dtype=str), embeddings=embeddings)
        result[name] = {sample: embedding for sample, embedding in zip(frame.sample_id, embeddings)}
        model.cpu()
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return result

def chimeric_registry(face_frame, finger_frame, partition, mapping_seed):
    face_ids = sorted(face_frame.loc[face_frame.partition == partition, "identity"].unique())
    finger_ids = sorted(finger_frame.loc[finger_frame.partition == partition, "identity"].unique())
    assert len(face_ids) == len(finger_ids) and len(face_ids) >= 2
    # A separate random stream for each partition; pairing never crosses splits.
    offset = {"model_val": 5000, "dev": 10000, "test": 20000}[partition]
    finger_ids = np.random.default_rng(mapping_seed + offset).permutation(finger_ids)
    return pd.DataFrame({"partition": partition, "chimeric_id": np.arange(len(face_ids)),
                         "face_identity": face_ids, "finger_identity": finger_ids})

def build_trials(face_frame, finger_frame, registry, embeddings):
    partition = registry.partition.iloc[0]
    assert registry.partition.nunique() == 1
    assert registry.face_identity.is_unique and registry.finger_identity.is_unique
    face_subset = face_frame[face_frame.partition == partition]
    finger_subset = finger_frame[finger_frame.partition == partition]
    assert set(registry.face_identity) == set(face_subset.identity)
    assert set(registry.finger_identity) == set(finger_subset.identity)
    n = len(registry)
    assert np.array_equal(registry.chimeric_id.to_numpy(), np.arange(n))
    face_ref = face_subset[face_subset.role == "reference"].set_index("identity").loc[registry.face_identity]
    face_probe = face_subset[face_subset.role == "probe"].set_index("identity").loc[registry.face_identity]
    finger_ref = finger_subset[finger_subset.role == "reference"].set_index("identity").loc[registry.finger_identity]
    assert len(face_ref) == len(face_probe) == len(finger_ref) == n
    query_index = np.repeat(np.arange(n), n)
    reference_index = np.tile(np.arange(n), n)
    genuine = query_index == reference_index
    face_distances = {}
    for key in ("face", "face_pretrained"):
        references = np.stack([embeddings[key][sample] for sample in face_ref.sample_id])
        probes = np.stack([embeddings[key][sample] for sample in face_probe.sample_id])
        face_distances[key] = cross_distances(probes, references).ravel()
    chunks = []
    for condition in CONDITIONS:
        probes = finger_subset[(finger_subset.role == "probe") & (finger_subset.condition == condition)]
        probes = probes.set_index("identity").loc[registry.finger_identity]
        assert len(probes) == n
        assert (face_ref.pixel_sha256.to_numpy() != face_probe.pixel_sha256.to_numpy()).all()
        assert (finger_ref.pixel_sha256.to_numpy() != probes.pixel_sha256.to_numpy()).all()
        fp_distances = cross_distances(
            np.stack([embeddings["finger"][sample] for sample in probes.sample_id]),
            np.stack([embeddings["finger"][sample] for sample in finger_ref.sample_id])).ravel()
        chunks.append(pd.DataFrame({
            "partition": partition, "condition": condition,
            "probe_identity": query_index, "reference_identity": reference_index, "genuine": genuine,
            "face_probe_identity": registry.face_identity.to_numpy()[query_index],
            "face_reference_identity": registry.face_identity.to_numpy()[reference_index],
            "finger_probe_identity": registry.finger_identity.to_numpy()[query_index],
            "finger_reference_identity": registry.finger_identity.to_numpy()[reference_index],
            "face_probe_id": face_probe.sample_id.to_numpy()[query_index],
            "face_reference_id": face_ref.sample_id.to_numpy()[reference_index],
            "finger_probe_id": probes.sample_id.to_numpy()[query_index],
            "finger_reference_id": finger_ref.sample_id.to_numpy()[reference_index],
            "face_distance": face_distances["face"], "face_pretrained_distance": face_distances["face_pretrained"],
            "finger_distance": fp_distances,
        }))
    trials = pd.concat(chunks, ignore_index=True)
    assert len(trials) == len(CONDITIONS) * n * n
    assert int(trials.genuine.sum()) == len(CONDITIONS) * n
    assert (trials.face_probe_id != trials.face_reference_id).all()
    assert (trials.finger_probe_id != trials.finger_reference_id).all()
    assert not trials.duplicated(["condition", "probe_identity", "reference_identity"]).any()
    return trials

selection_dev_started = time.perf_counter()
selection_embeddings = cache_partition_embeddings("model_val")
selection_trials = {}
for mapping_seed in CFG["MAPPING_SEEDS"]:
    registry = chimeric_registry(face_df, finger_df, "model_val", mapping_seed)
    registry.to_csv(OUT / "manifests" / f"model_val_mapping_{mapping_seed}.csv", index=False)
    trials = build_trials(face_df, finger_df, registry, selection_embeddings)
    trials.to_csv(OUT / "results" / f"model_val_trials_{mapping_seed}.csv.gz", index=False)
    selection_trials[mapping_seed] = trials

dev_embeddings = cache_partition_embeddings("dev")
dev_trials = {}
for mapping_seed in CFG["MAPPING_SEEDS"]:
    registry = chimeric_registry(face_df, finger_df, "dev", mapping_seed)
    registry.to_csv(OUT / "manifests" / f"dev_mapping_{mapping_seed}.csv", index=False)
    trials = build_trials(face_df, finger_df, registry, dev_embeddings)
    trials.to_csv(OUT / "results" / f"dev_trials_{mapping_seed}.csv.gz", index=False)
    dev_trials[mapping_seed] = trials
print("Development trials per mapping:", len(next(iter(dev_trials.values()))))


## 9. Select fusion on model validation; freeze thresholds on development

Compare both face recipes, fingerprint, and raw/tuned fusion for each face.
Normalization and a prespecified 21-weight grid use model-validation identities.
The default objective maximizes TAR at the FAR budget, then minimizes EER,
retaining the original preference for 0.5 on a tie. The search exports tail
separation and all primary-optimal candidates; a tie is not a unique optimum.
A tail-gap fusion tie-break is available as an explicit experiment. It worsened
FAR in the archived development-tuned replay, so it is not enabled by default.
Development identities only calibrate the frozen score functions' thresholds.
The separate conservative operating point halves the unused gap between the
largest accepted development genuine score and the FAR-budget threshold.
Report its test FAR AND TAR beside the budget threshold; do not pick whichever
result looks best on test. No claim of improved performance is made in advance.


In [ ]:
def method_scores(trials, method):
    kind = method["kind"]
    if kind == "face_pretrained":
        return trials.face_pretrained_distance.to_numpy(dtype=np.float64)
    if kind == "face":
        return trials.face_distance.to_numpy(dtype=np.float64)
    if kind == "finger":
        return trials.finger_distance.to_numpy(dtype=np.float64)
    face_column = method.get("face_column", "face_distance")
    assert face_column in {"face_distance", "face_pretrained_distance"}
    face = trials[face_column].to_numpy(dtype=np.float64)
    finger = trials.finger_distance.to_numpy(dtype=np.float64)
    if kind == "z_fusion":
        face = (face - method["face_mean"]) / method["face_std"]
        finger = (finger - method["finger_mean"]) / method["finger_std"]
    else:
        assert kind == "raw_fusion"
    return (1 - method["finger_weight"]) * face + method["finger_weight"] * finger

def fit_fusion(trials, target_far, objective, face_column="face_distance", tiebreak="center"):
    # The notebook uses model_val. dev is supported only for explicitly labelled
    # exploratory replay of old archives that have no model-validation scores.
    assert set(trials.partition) in ({"model_val"}, {"dev"}), "Fusion fitting must never see test trials."
    assert objective in {"eer", "tar_at_target_far"}
    assert tiebreak in {"center", "tail_gap"}
    assert face_column in {"face_distance", "face_pretrained_distance"}
    genuine = trials.genuine.to_numpy(dtype=bool)
    normalization = {}
    for modality, column in (("face", face_column), ("finger", "finger_distance")):
        impostor = trials.loc[~genuine, column].to_numpy(dtype=np.float64)
        normalization[modality + "_mean"] = float(impostor.mean())
        deviation = float(impostor.std())
        if deviation < 1e-8:
            raise ValueError(f"Collapsed {modality} selection impostor scores; inspect embeddings.")
        normalization[modality + "_std"] = deviation
    candidates = []
    for weight in np.linspace(0, 1, 21):
        method = {"kind": "z_fusion", "finger_weight": float(weight), "face_column": face_column,
                  "selection_partition": str(trials.partition.iloc[0]), "selection_tiebreak": tiebreak,
                  **normalization}
        scores = method_scores(trials, method)
        diagnostics = score_diagnostics(scores, genuine, target_far)
        candidates.append((method, diagnostics))
    def primary_key(row):
        return tuple(round(value, 12) for value in selection_key(row[1], objective)[:2])
    def candidate_key(row):
        center, tail = abs(row[0]["finger_weight"] - 0.5), -row[1]["tail_gap"]
        return (*primary_key(row), *((center, tail) if tiebreak == "center" else (tail, center)))
    chosen = min(candidates, key=candidate_key)
    search = pd.DataFrame([{"face_column": face_column, "finger_weight": method["finger_weight"],
                            "selected": method is chosen[0],
                            "primary_optimal": primary_key((method, diagnostics)) == primary_key(chosen),
                            **diagnostics}
                           for method, diagnostics in candidates])
    chosen[0]["primary_tie_count"] = int(search.primary_optimal.sum())
    return chosen[0], search

def calibrate_methods(trials, target_far, objective, selection_trials, report_conservative=False, fusion_tiebreak="center"):
    assert set(trials.partition) == {"dev"}, "Calibration must never see test trials."
    assert set(selection_trials.partition) == {"model_val"}, "The new protocol selects fusion on model_val."
    for modality in ("face", "finger"):
        columns = [f"{modality}_{role}_identity" for role in ("probe", "reference")]
        calibration_ids = set(trials[columns].to_numpy().ravel())
        selection_ids = set(selection_trials[columns].to_numpy().ravel())
        assert not calibration_ids & selection_ids, f"Fusion selection/calibration identity overlap: {modality}"
    genuine = trials.genuine.to_numpy(dtype=bool)
    methods = {"face_pretrained": {"kind": "face_pretrained"},
               "face_selected": {"kind": "face"}, "fingerprint": {"kind": "finger"}}
    searches = []
    for prefix, face_column in (("fusion", "face_distance"), ("fusion_pretrained", "face_pretrained_distance")):
        methods[prefix + "_equal_raw"] = {"kind": "raw_fusion", "finger_weight": 0.5, "face_column": face_column}
        methods[prefix + "_original_raw"] = {"kind": "raw_fusion", "finger_weight": 0.75, "face_column": face_column}
        methods[prefix + "_tuned"], search = fit_fusion(selection_trials, target_far, objective, face_column, fusion_tiebreak)
        searches.append(search)
    if report_conservative:
        for name in ("fingerprint", "fusion_tuned", "fusion_pretrained_tuned"):
            methods[name + "_conservative"] = {**methods[name], "threshold_policy": "genuine_midpoint"}
    for method in methods.values():
        scores = method_scores(trials, method)
        method.setdefault("threshold_policy", "far_budget")
        method["far_budget_threshold"] = threshold_for_far(scores[~genuine], target_far)
        method["threshold"] = calibrated_threshold(scores, genuine, target_far, method["threshold_policy"])
        method["development"] = {**roc_summary(scores, genuine),
                                 **operating_metrics(scores, genuine, method["threshold"])}
        assert method["development"]["far"] <= target_far + 1e-12
        method["target_far"] = target_far
    return methods, pd.concat(searches, ignore_index=True)

record_stage("selection_dev_embeddings_and_trials", selection_dev_started, synchronize_cuda=True)
calibration_started = time.perf_counter()
calibrations = {}
for mapping_seed, trials in dev_trials.items():
    methods, search = calibrate_methods(trials, CFG["TARGET_FAR"], CFG["FUSION_OBJECTIVE"],
                                        selection_trials[mapping_seed], CFG["REPORT_CONSERVATIVE_THRESHOLD"], CFG["FUSION_TIEBREAK"])
    payload = {"mapping_seed": mapping_seed, "calibration_partition": "dev", "methods": methods,
               "fusion_selection_partition": "model_val",
               "fusion_tiebreak": CFG["FUSION_TIEBREAK"],
               "selection_trial_sha256": sha256_file(OUT / "results" / f"model_val_trials_{mapping_seed}.csv.gz"),
               "objective": CFG["FUSION_OBJECTIVE"], "primary_condition": CFG["PRIMARY_CONDITION"],
               "trial_sha256": sha256_file(OUT / "results" / f"dev_trials_{mapping_seed}.csv.gz")}
    write_json(OUT / "calibration" / f"mapping_{mapping_seed}.json", payload)
    search.to_csv(OUT / "calibration" / f"weight_search_{mapping_seed}.csv", index=False)
    calibrations[mapping_seed] = payload
    print("Mapping", mapping_seed, "model-val-selected fingerprint weight:", methods["fusion_tuned"]["finger_weight"],
          "| primary-optimal weights:", methods["fusion_tuned"]["primary_tie_count"])
record_stage("calibration", calibration_started)
calibration_hashes = {seed: sha256_file(OUT / "calibration" / f"mapping_{seed}.json")
                      for seed in CFG["MAPPING_SEEDS"]}
write_json(OUT / "calibration" / "frozen_sha256.json", {str(k): v for k, v in calibration_hashes.items()})
print("Calibration saved and frozen. The following cells only evaluate test identities.")


## 10. Test with frozen settings and identity-level uncertainty

Each test condition uses the development threshold without recalibration.
The primary result pools the nine conditions equally; condition-specific
results reveal changes that an overall EER can hide.

The bootstrap resamples chimeric identities as nodes: genuine trials receive
their identity multiplicity, and impostor trials receive the product of their
two different identity multiplicities. This keeps repeated images/conditions
together and excludes artificial impostor comparisons between copies of the
same identity. Paired FAR and TAR differences use the same resamples for all methods.
Intervals are approximate and conditional on this trained model, calibration,
and mapping; they exclude training/calibration uncertainty. EER/AUC are point
summaries, while the intervals below describe FAR and TAR at fixed thresholds.
If no false accepts are observed, a [0, 0] bootstrap FAR interval is not proof
of zero population FAR; report the counts and avoid a security guarantee.


In [ ]:
def identity_bootstrap_operating(trials, distances, threshold, multiplicities):
    genuine = trials.genuine.to_numpy(dtype=bool)
    query = trials.probe_identity.to_numpy(dtype=int)
    reference = trials.reference_identity.to_numpy(dtype=int)
    n = multiplicities.shape[1]
    assert query.max() < n and reference.max() < n
    accepted = np.asarray(distances, dtype=np.float64) <= threshold
    genuine_total = np.bincount(query[genuine], minlength=n).astype(float)
    genuine_accept = np.bincount(query[genuine], weights=accepted[genuine].astype(float), minlength=n)
    pair = query[~genuine] * n + reference[~genuine]
    impostor_total = np.bincount(pair, minlength=n*n).reshape(n, n).astype(float)
    impostor_accept = np.bincount(pair, weights=accepted[~genuine].astype(float), minlength=n*n).reshape(n, n)
    assert not np.diag(impostor_total).any()
    genuine_denominator = multiplicities @ genuine_total
    impostor_denominator = np.einsum("bi,ij,bj->b", multiplicities, impostor_total, multiplicities, optimize=True)
    valid = (genuine_denominator > 0) & (impostor_denominator > 0)
    tar = np.full(len(multiplicities), np.nan)
    far = np.full(len(multiplicities), np.nan)
    tar[valid] = (multiplicities @ genuine_accept)[valid] / genuine_denominator[valid]
    numerator = np.einsum("bi,ij,bj->b", multiplicities, impostor_accept, multiplicities, optimize=True)
    far[valid] = numerator[valid] / impostor_denominator[valid]
    return tar, far

def interval(samples):
    finite = np.asarray(samples)[np.isfinite(samples)]
    if not len(finite):
        raise ValueError("No valid identity bootstrap replicates; too few evaluation identities.")
    return [float(value) for value in np.quantile(finite, [0.025, 0.975])]

def evaluate_test(trials, calibration, mapping_seed, bootstrap_replicates):
    assert set(trials.partition) == {"test"}, "Final results require the test partition."
    assert calibration["calibration_partition"] == "dev"
    n = trials.probe_identity.nunique()
    multiplicities = np.random.default_rng(mapping_seed + 30000).multinomial(
        n, np.repeat(1 / n, n), size=bootstrap_replicates).astype(float)
    results, draws, point_metrics, scored = [], {}, {}, trials.copy()
    for name, method in calibration["methods"].items():
        scores = method_scores(trials, method)
        scored[name + "_score"] = scores
        scored[name + "_accepted"] = scores <= method["threshold"]
        for condition in ["All"] + CONDITIONS:
            mask = np.ones(len(trials), dtype=bool) if condition == "All" else trials.condition.to_numpy() == condition
            subset = trials.loc[mask]
            values = scores[mask]
            genuine = subset.genuine.to_numpy(dtype=bool)
            metrics = {**roc_summary(values, genuine), **operating_metrics(values, genuine, method["threshold"])}
            tar_samples, far_samples = identity_bootstrap_operating(subset, values, method["threshold"], multiplicities)
            tar_ci, far_ci = interval(tar_samples), interval(far_samples)
            results.append({"mapping_seed": mapping_seed, "method": name, "condition": condition,
                            "identities": n, "threshold": method["threshold"], "target_far": method["target_far"],
                            "threshold_policy": method.get("threshold_policy", "far_budget"),
                            "tar_ci_low": tar_ci[0], "tar_ci_high": tar_ci[1],
                            "far_ci_low": far_ci[0], "far_ci_high": far_ci[1],
                            "no_observed_false_accepts": metrics["false_accepts"] == 0, **metrics})
            if condition == "All":
                draws[name] = {"tar": tar_samples, "far": far_samples}
                point_metrics[name] = metrics
    differences = []
    comparisons = [("fusion_tuned", baseline) for baseline in
                   ("face_selected", "fingerprint", "fusion_equal_raw", "fusion_pretrained_tuned")
                   if baseline in draws]
    comparisons += [(name + "_conservative", name) for name in
                    ("fingerprint", "fusion_tuned", "fusion_pretrained_tuned") if name + "_conservative" in draws]
    comparisons += [("fusion_tuned_conservative", "fingerprint_conservative"),
                    ("fusion_pretrained_tuned_conservative", "fingerprint_conservative"),
                    ("fusion_tuned_conservative", "fusion_pretrained_tuned_conservative")]
    comparisons = [(method, baseline) for method, baseline in comparisons if method in draws and baseline in draws]
    for method, baseline in comparisons:
        row = {"mapping_seed": mapping_seed, "comparison": method + " minus " + baseline,
               "note": "Differences are rates; multiply by 100 for percentage points. Each threshold is frozen on development."}
        for metric in ("tar", "far"):
            low, high = interval(draws[method][metric] - draws[baseline][metric])
            row.update({metric + "_difference": point_metrics[method][metric] - point_metrics[baseline][metric],
                        metric + "_difference_ci_low": low, metric + "_difference_ci_high": high})
        differences.append(row)
    return pd.DataFrame(results), pd.DataFrame(differences), scored

# Calibration is read back from disk and checked before test extraction.
for mapping_seed in CFG["MAPPING_SEEDS"]:
    path = OUT / "calibration" / f"mapping_{mapping_seed}.json"
    assert sha256_file(path) == calibration_hashes[mapping_seed]
    calibrations[mapping_seed] = json.loads(path.read_text())

test_started = time.perf_counter()
test_embeddings = cache_partition_embeddings("test")
result_tables, difference_tables = [], []
first_scored_test = None
for mapping_seed in CFG["MAPPING_SEEDS"]:
    registry = chimeric_registry(face_df, finger_df, "test", mapping_seed)
    registry.to_csv(OUT / "manifests" / f"test_mapping_{mapping_seed}.csv", index=False)
    trials = build_trials(face_df, finger_df, registry, test_embeddings)
    results, differences, scored = evaluate_test(trials, calibrations[mapping_seed], mapping_seed,
                                                 CFG["BOOTSTRAP_REPLICATES"])
    results.to_csv(OUT / "results" / f"test_metrics_{mapping_seed}.csv", index=False)
    differences.to_csv(OUT / "results" / f"paired_operating_differences_{mapping_seed}.csv", index=False)
    scored.to_csv(OUT / "results" / f"test_trials_scores_{mapping_seed}.csv.gz", index=False)
    result_tables.append(results)
    difference_tables.append(differences)
    if first_scored_test is None:
        first_scored_test = scored
    assert sha256_file(OUT / "calibration" / f"mapping_{mapping_seed}.json") == calibration_hashes[mapping_seed]

record_stage("test_embeddings_trials_and_bootstrap", test_started, synchronize_cuda=True)
test_results = pd.concat(result_tables, ignore_index=True)
paired_differences = pd.concat(difference_tables, ignore_index=True)
test_results.to_csv(OUT / "results" / "test_metrics_all_mappings.csv", index=False)
paired_differences.to_csv(OUT / "results" / "paired_operating_differences.csv", index=False)
primary = test_results[test_results.condition == "All"].copy()
columns = ["mapping_seed", "method", "eer", "auc", "far", "far_ci_low", "far_ci_high", "tar", "tar_ci_low", "tar_ci_high",
           "false_accepts", "impostor_trials", "false_rejects", "genuine_trials"]
display(primary[columns].style.format({key: "{:.6%}" for key in
                                      ("eer", "far", "far_ci_low", "far_ci_high", "tar", "tar_ci_low", "tar_ci_high")}
                                      | {"auc": "{:.4f}"}))
display(paired_differences)
mapping_variability = primary.groupby("method")[["eer", "tar", "far"]].agg(["mean", "std", "min", "max"])
mapping_variability.to_csv(OUT / "results" / "mapping_sensitivity.csv")
print("Mapping sensitivity on the SAME test cohort (not independent training runs):")
display(mapping_variability)
print("Inspect achieved FAR alongside TAR. Interpolated EER is a curve summary, not threshold calibration.")
print("Face genuine acquisition pairs:", CFG["N_TEST"], "| fingerprint alteration pairs:", CFG["N_TEST"] * len(CONDITIONS))
print("Zero observed false accepts and a collapsed bootstrap interval do not imply zero population FAR.")
print("Conservative thresholds are separate prespecified operating points; compare FAR and TAR together.")


## 11. Plots and export

Plots use the first prespecified mapping; all mappings and conditions are in
the CSVs. The ZIP includes image/split/trial manifests, checkpoints, embedding
caches, frozen calibration, training histories, environment details, figures,
and metrics. Download it from Kaggle's Output panel and save an executed
notebook version alongside it. This notebook does not fabricate missing run
outputs or claim improvement before the test executes.


In [ ]:
plot_seed = CFG["MAPPING_SEEDS"][0]
plot_methods = ("face_pretrained", "face_selected", "fingerprint", "fusion_equal_raw", "fusion_tuned")
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
genuine = first_scored_test.genuine.to_numpy(dtype=bool)
for name in plot_methods:
    scores = first_scored_test[name + "_score"].to_numpy()
    fpr, tpr, _ = roc_curve(genuine.astype(int), -scores)
    axes[0].plot(fpr, tpr, label=name)
axes[0].set(xlabel="False acceptance rate", ylabel="True acceptance rate", title="Test ROC (descriptive)")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.2)

plot_rows = primary[primary.mapping_seed == plot_seed].set_index("method").loc[list(plot_methods)]
x = np.arange(len(plot_rows))
axes[1].bar(x, 100 * plot_rows.eer)
axes[1].set(xticks=x, xticklabels=plot_rows.index, ylabel="EER (%)", title="Test EER")
axes[1].tick_params(axis="x", rotation=70)

hardness = test_results[(test_results.mapping_seed == plot_seed) & (test_results.method == "fusion_tuned")
                        & (test_results.condition != "All")].set_index("condition").loc[CONDITIONS]
x = np.arange(len(hardness))
axes[2].plot(x, 100 * hardness.tar, "o-", label="TAR")
axes[2].fill_between(x, 100 * hardness.tar_ci_low, 100 * hardness.tar_ci_high, alpha=0.2, label="identity bootstrap 95%")
axes[2].set(xticks=x, xticklabels=hardness.index, ylabel="TAR (%)", title="Fixed-threshold alteration robustness")
axes[2].tick_params(axis="x", rotation=70)
axes[2].legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT / "results" / "test_overview.png", dpi=160, bbox_inches="tight")
plt.show()


# Keep the ordinary operating points, and show conservative comparisons beside
# a fingerprint baseline using the same threshold policy.
operating_names = ["fingerprint", "fingerprint_conservative", "fusion_tuned",
                   "fusion_tuned_conservative", "fusion_pretrained_tuned",
                   "fusion_pretrained_tuned_conservative"]
operating_names = [name for name in operating_names if name in set(primary.method)]
operating = primary[primary.method.isin(operating_names)].copy()
display(operating[["mapping_seed", "method", "false_accepts", "impostor_trials", "far",
                   "far_ci_low", "far_ci_high", "false_rejects", "genuine_trials", "tar"]])
fig, ax = plt.subplots(figsize=(12, 5))
for offset, seed in enumerate(CFG["MAPPING_SEEDS"]):
    rows = operating[operating.mapping_seed == seed].set_index("method").loc[operating_names]
    width = 0.8 / len(CFG["MAPPING_SEEDS"])
    ax.bar(np.arange(len(rows)) + offset * width, 100 * rows.far, width, label=f"Mapping {seed}")
ax.axhline(100 * CFG["TARGET_FAR"], color="gray", linestyle="--", label="Development FAR target")
ax.set(xticks=np.arange(len(operating_names)) + width * (len(CFG["MAPPING_SEEDS"]) - 1) / 2,
       xticklabels=operating_names, ylabel="Achieved test FAR (%)", title="Frozen threshold operating points")
ax.tick_params(axis="x", rotation=35)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "results" / "operating_points.png", dpi=160, bbox_inches="tight")
plt.show()
write_json(OUT / "inventory_cache_provenance.json", inventory_cache_events)
print("A 0% FAR with [0, 0] bootstrap bounds records no observed errors; it is not a population upper bound.")

run_notes = {
    "optimization_version": "2026-11-11-io-reporting-v2",
    "inventory_cache": inventory_cache_events,
    "stage_timings": stage_timings,
    "decoded_cache_bytes_at_completion": decoded_image_cache_bytes,
    "runtime_note": "NUM_WORKERS=0 changes augmentation RNG scheduling; this is a new training run, not bitwise replay",
    "protocol": "Synthetic LFW/SOCOFing chimeric reference-to-probe verification",
    "identity_separation": "fit / model_val / dev / test, separately in each modality",
    "scope": "Unseen during task-specific fitting; pretraining identity overlap unverified",
    "fingerprint_scope": "One Real source capture plus nine synthetic alterations per selected finger",
    "uncertainty": "Identity-node bootstrap for fixed-threshold FAR/TAR; conditional on trained model and calibration",
    "mapping_sensitivity": "Same biological test cohort across mappings; not independent training repeats",
    "primary_condition": "All nine fingerprint alterations, equally represented",
    "model_selection": "Model-validation TAR at target FAR, then EER, then normalized tail separation (default)",
    "fusion_selection": "Model-validation identities select normalization and weights after encoder selection",
    "calibration": "Separate development identities calibrate thresholds only; far_budget and optional genuine_midpoint policies",
    "followup_status": "Exploratory follow-up on the previously inspected split; requires fresh data for confirmation",
    "presentation_attack_evidence": "None; no liveness or physical spoof-resistance claim",
    "face_selection": face_selection,
    "finger_selection": finger_selection,
}
write_json(OUT / "run_notes.json", run_notes)
# The builder embeds the exact source snapshot in the setup cell.
if "OPTIMIZED_SOURCE_TEXT" in globals():
    (OUT / "optimized_notebook_source.py").write_text(OPTIMIZED_SOURCE_TEXT)
    write_json(OUT / "source_provenance.json", {"sha256": sha256_bytes(OPTIMIZED_SOURCE_TEXT.encode()),
                                               "build": "2026-11-11-io-reporting-v2"})
cache_output = Path(CFG["INVENTORY_CACHE_DIR"])
if cache_output.is_dir():
    shutil.make_archive(str(cache_output), "zip", root_dir=cache_output)
    print("Reusable inventory cache:", str(cache_output) + ".zip")
integrity = {str(path.relative_to(OUT)): sha256_file(path) for path in sorted(OUT.rglob("*")) if path.is_file()}
write_json(OUT / "run_integrity_sha256.json", integrity)
archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
print("Completed run:", OUT)
print("Download archive:", archive)
print("Save a Kaggle notebook version with all executed outputs for the research record.")
